In [ ]:
#@title DeFooocus
#@markdown **Launch the interface DeFocus (Fooocus fork)** | You need to connect with T4/A100/V100

theme = "dark" #@param ["dark", "light"]
preset = "deafult" #@param ["deafult", "realistic", "anime", "lcm", "sai", "turbo", "lighting", "hypersd", "playground_v2.5", "dpo", "spo", "sd1.5"]
advenced_args = "--share --attention-split --always-high-vram --disable-offload-from-vram --all-in-fp16" #@param {type: "string"}

import os, re, importlib.util
print("[DeFooocus] Preparing ...")
args = f"{advenced_args} --theme {theme}" + (f" --preset {preset}" if preset != "deafult" else "")

# 0. Remove leftovers from an interrupted pip uninstall
!rm -rf /usr/local/lib/python3.13/dist-packages/~ransformers*

# 1. Code
!pip install -q pygit2
%cd /content
![ -d DeFooocus ] || git clone https://github.com/ehristoforu/DeFooocus.git
%cd /content/DeFooocus

# 2. Dependencies (drop pins that don't work on Python 3.13, then fix versions at the end)
!sed -i '/^pygit2/d;/^numpy/d;/^scipy/d;/^onnxruntime/d;/^rembg/d;/^safetensors/d;/^tokenizers/d;/^pyyaml/d;/^transformers/d' requirements_versions.txt
!pip install -q "torchsde==0.2.6" rembg
!pip install -q -r requirements_versions.txt
!pip install -q "transformers>=4.46,<5" "numpy>=2.1.0"

# 3. Patch bundled BLIP (image describe) for transformers 4.5x
!git -C /content/DeFooocus checkout -- extras/BLIP/models/med.py
_p = "/content/DeFooocus/extras/BLIP/models/med.py"
_src = open(_p).read()
_src = re.sub(
    r"from transformers\.modeling_utils import \((.*?)\)",
    '''from transformers.modeling_utils import PreTrainedModel
from transformers import GenerationMixin
from transformers.pytorch_utils import apply_chunking_to_forward, prune_linear_layer
try:
    from transformers.pytorch_utils import find_pruneable_heads_and_indices
except ImportError:
    def find_pruneable_heads_and_indices(heads, n_heads, head_size, already_pruned_heads):
        mask = torch.ones(n_heads, head_size)
        heads = set(heads) - already_pruned_heads
        for head in heads:
            head = head - sum(1 if h < head else 0 for h in already_pruned_heads)
            mask[head] = 0
        mask = mask.view(-1).contiguous().eq(1)
        index = torch.arange(len(mask))[mask].long()
        return heads, index''',
    _src, count=1, flags=re.DOTALL)
_src = _src.replace("class BertLMHeadModel(BertPreTrainedModel):",
                    "class BertLMHeadModel(BertPreTrainedModel, GenerationMixin):")
open(_p, "w").write(_src)

# 4. Patch gradio 3.x for Starlette 1.x (TemplateResponse now takes request first)
_gr = os.path.dirname(importlib.util.find_spec("gradio").origin)
_rp = os.path.join(_gr, "routes.py")
_s = open(_rp).read()
_s = re.sub(r'TemplateResponse\(\s*(\w+),\s*\{"request": request',
            r'TemplateResponse(request, \1, {"request": request', _s)
open(_rp, "w").write(_s)
!grep -n -A1 "TemplateResponse(" {_rp}

# 5. NumPy 1.x compatibility for gradio 3.x
import numpy
_site = os.path.dirname(os.path.dirname(numpy.__file__))
open(os.path.join(_site, "numpy2_compat.py"), "w").write('''
import numpy as np

def _obj2sctype(rep, default=None):
    if isinstance(rep, type) and issubclass(rep, np.generic):
        return rep
    try:
        return np.dtype(rep).type
    except Exception:
        return default

_aliases = {
    "float_": np.float64, "bool8": np.bool_, "int0": np.intp, "uint0": np.uintp,
    "complex_": np.complex128, "cfloat": np.complex128, "unicode_": np.str_,
    "string_": np.bytes_, "str0": np.str_, "bytes0": np.bytes_, "object0": np.object_,
    "void0": np.void, "longfloat": np.longdouble, "singlecomplex": np.complex64,
    "Inf": np.inf, "NaN": np.nan, "infty": np.inf, "PINF": np.inf, "NINF": -np.inf,
    "obj2sctype": _obj2sctype,
    "issctype": lambda rep: isinstance(rep, type) and issubclass(rep, np.generic),
    "sctype2char": lambda s: np.dtype(s).char,
    "find_common_type": lambda a, s=(): np.result_type(*a, *s),
    "round_": np.round, "product": np.prod, "cumproduct": np.cumprod,
    "sometrue": np.any, "alltrue": np.all,
    "asfarray": lambda a, dtype=np.float64: np.asarray(a, dtype=dtype),
}
for _k, _v in _aliases.items():
    if not hasattr(np, _k):
        setattr(np, _k, _v)
''')
open(os.path.join(_site, "zz_numpy2_compat.pth"), "w").write("import numpy2_compat\n")
!python -c "import numpy as np; print(np.obj2sctype(np.uint8), np.float_, np.bool8)"

# 6. Launch
print("[DeFooocus] Starting ...")
!python entry_with_update.py $args

[DeFooocus] Preparing ...
/content
/content/DeFooocus
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 3.41.2 requires numpy~=1.0, but you have numpy 2.5.3 which is incompatible.
gradio 3.41.2 requires pillow<11.0,>=8.0, but you have pillow 12.3.0 which is incompatible.
hf-gradio 0.4.1 requires gradio-client<3.0,>=2.0, but you have gradio-client 0.5.0 which is incompatible.
bigquery-magics 0.14.0 requires packaging>=24.2.0, but you have packaging 24.1 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
dataproc-spark-connect 1.1.0 requires websockets>=14.0, but you have websockets 11.0.3 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency